# Mini-DSKG: ML Dataset Knowledge Graph

A knowledge graph is a graph **plus semantics**: nodes are entities, edges are relations, both carry attributes. DigitalSreeni’s NetworkX tutorial builds a Python *learning path*. DSKG (Färber & Lamprecht, 2021) builds a *dataset–paper* graph by string-matching dataset names in abstracts.

This notebook does DSKG’s construction at student scale, with the NetworkX analysis from the tutorial.

- Catalog: `data/datasets.csv` (~78 ML datasets, aliases, task, license)
- Papers: OpenAlex abstracts (successor to Microsoft Academic / MAG)
- Match: DSKG-style alias match + stopword filter
- Graph: Dataset, Paper, Author, Task in NetworkX, saved as GraphML

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(ROOT / "src"))

from match_mentions import load_catalog, match_papers, write_mentions, MENTIONS_PATH, PAPERS_PATH
from fetch_papers import fetch_all
from build_graph import build_graph, load_papers, load_mentions, GRAPHML_PATH
from analyze import analyze, by_name, similar_to, problem_search, mention_counts, dataset_h_index, paper_neighbors
from visualize import build_network, HTML_PATH
import networkx as nx
from matplotlib import pyplot as plt

## 1. Seed catalog

DSKG started from OpenAIRE + Wikidata. We start from a hand-curated ML catalog with **aliases** (the strings we will look for in abstracts).

In [ ]:
catalog = load_catalog()
print(f"{len(catalog)} datasets, {len({r['task'] for r in catalog})} tasks")
for row in catalog[:8]:
    print(f"  {row['id']:16s}  {row['task']:24s}  aliases={row['aliases']}")

## 2. Fetch papers and match mentions

OpenAlex search per dataset name, then match **every** alias against **every** paper so a COCO+ImageNet paper becomes a bridge. Skip cached fetches.

In [ ]:
if not PAPERS_PATH.exists():
    fetch_all(per_page=25)
papers = load_papers()
print(f"{len(papers)} papers in {PAPERS_PATH}")

mentions = match_papers(list(papers.values()), catalog)
write_mentions(mentions)
print(f"{len(mentions)} mentions, {len({m['dataset_id'] for m in mentions})} datasets linked")

## 3. Build the knowledge graph

Same pattern as the tutorial: add nodes with attributes, then labeled edges (`mentioned`, `writtenBy`, `usedFor`, `similarTo`). Persist with GraphML.

In [ ]:
G = build_graph(catalog, papers, mentions)
nx.write_graphml(G, GRAPHML_PATH)
kinds = {}
for _, d in G.nodes(data=True):
    kinds[d.get("kind")] = kinds.get(d.get("kind"), 0) + 1
print(kinds, "edges", G.number_of_edges())
print("saved", GRAPHML_PATH)

## 4. Analysis (tutorial metrics, DSKG questions)

| Tutorial | Mini-DSKG question |
| --- | --- |
| degree | most-mentioned datasets |
| h-index of datasets (DSKG §6) | h papers each with ≥ h citations |
| betweenness | datasets that bridge research clusters |
| communities | vision vs NLP vs speech |
| Jaccard | datasets similar to ImageNet |
| descendants / “remove a station” | papers that depend on a dataset |

In [ ]:
stats = analyze(G)
print(f"{stats['n_datasets']} datasets · {stats['n_papers']} papers · {stats['n_authors']} authors")
print("\nTop mentions")
for row in stats["top_mentions"][:10]:
    print(f"  {row['mentions']:4d}  h={row['h_index']:3d}  {row['name']}")
print("\nBetweenness (dataset projection)")
for row in stats["top_betweenness"][:8]:
    print(f"  {row['betweenness']:.4f}  {row['name']}")
print("\nCommunities", stats["n_communities"])
for i, c in enumerate(stats["communities"][:5], 1):
    print(f"  {i}: {', '.join(c[:8])}")

In [ ]:
top = stats["top_mentions"][:12]
plt.figure(figsize=(9, 4))
plt.barh([r["name"] for r in reversed(top)], [r["mentions"] for r in reversed(top)], color="#4C78A8")
plt.xlabel("papers mentioning the dataset")
plt.title("Most mentioned datasets")
plt.tight_layout()
plt.show()

In [ ]:
imagenet = by_name(G, "imagenet")
print("Similar to ImageNet (Jaccard on shared papers):")
for n, j in similar_to(G, imagenet):
    print(f"  {j:.3f}  {G.nodes[n]['name']}")

print("\nProblem search: 'object detection in autonomous driving'")
for n, s in problem_search(G, "object detection in autonomous driving"):
    print(f"  {s:6.0f}  {G.nodes[n]['name']}")

print(f"\nIf ImageNet disappeared, {len(paper_neighbors(G, imagenet))} papers in this graph mention it.")
print("sample:", [G.nodes[p]["name"][:70] for p in paper_neighbors(G, imagenet)[:5]])

## 5. Interactive explorer

PyVis HTML (what the tutorial installs but defers). Subgraph: all datasets and tasks, papers that mention two or more datasets, plus a few highly cited papers per dataset.

In [ ]:
net = build_network(G)
HTML_PATH.parent.mkdir(parents=True, exist_ok=True)
HTML_PATH.write_text(net.generate_html(notebook=False), encoding="utf-8")
print("open", HTML_PATH)